# Extract features

Caches what the models consume for every chart in a split, under `features/<extractor.id>/`:

- `songs/<audio_id>/mel.npy`: 80-band log-mel, one row per `frame_ms` of audio, computed once per audio file.
- `songs/<audio_id>/labels/<chart_id>.npy`: 7-class one-hot labels on the same frames (none, don, ka, DON, KA, drumroll, denden).
- `normalization/<split>.npz`: per-band mean/std from the split's **train** songs only, plus a hash of the manifest it was fit on.

Extraction is deterministic and skips files that already exist, so a new split only computes new songs plus its normalization. Windows are not stored: `TaikoDataset` cuts them on the fly, so window sizes can change without re-extracting.

In [ ]:
from donderformer.paths import DataPaths
from donderformer.data.feature_extractor import FeatureExtractor

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for the toy dataset.
paths = DataPaths()
SPLIT = "oni_v1"
extractor = FeatureExtractor(frame_ms=23)
print(extractor.id, "->", extractor.dir(paths))

In [ ]:
summary = extractor.extract(paths, SPLIT)

## Visual alignment check

The first ~6 seconds of one chart. Cyan lines are frames with a note label; they should sit on onsets in the spectrogram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from donderformer.data.data_splitter import read_manifest

row = read_manifest(paths, SPLIT, "train")[0]
mean, std = extractor.load_normalization(paths, SPLIT)
mel = (np.load(extractor.mel_path(paths, row["audio_id"])) - mean) / (std + 1e-8)
labels = np.load(extractor.labels_path(paths, row["audio_id"], row["chart_id"]))

plt.figure(figsize=(14, 4))
plt.imshow(mel[:260].T, origin="lower", aspect="auto", cmap="magma")
for frame, cls in enumerate(labels[:260].argmax(axis=1)):
    if cls: plt.axvline(frame, color="cyan", alpha=.65, linewidth=1)
plt.title(f"{row['chart_id']}: first ~6 seconds")
plt.xlabel(f"{extractor.frame_ms} ms frame"); plt.ylabel("Mel band"); plt.show()